# La Flamme Rouge — Scraping avec `undetected-chromedriver`

In [ ]:
!pip install undetected-chromedriver bs4 tqdm requests

In [ ]:
import os
import re
import time
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

WORK_DIR = '...'
GPX_DIR = os.path.join(WORK_DIR, 'data', '...')
ELE_DIR = os.path.join(WORK_DIR, 'data', '...')
os.makedirs(GPX_DIR, exist_ok=True)
os.makedirs(ELE_DIR, exist_ok=True)
print('GPX bruts      :', GPX_DIR)
print('GPX + élévation:', ELE_DIR)

In [ ]:
def get_elevation(file):
    """Enrichit un fichier GPX avec les données d'élévation via gpsvisualizer."""
    files = {'uploaded_file_1': open(file, 'rb')}
    formdata = {
        'convert_format': 'gpx',
        'add_elevation': 'auto',
        'profile_x': 'distance',
        'profile_y': 'altitude'
    }
    data = requests.post('https://www.gpsvisualizer.com/convert?output_elevation', files=files, data=formdata)
    result = re.findall(r'\/download\/convert\/[0-9\-]+data\.gpx', data.text)
    return 'https://www.gpsvisualizer.com' + result[0] if result else None


def get_gpx(race_id, driver):
    """
    Télécharge le GPX en réutilisant les cookies du driver
    pour contourner Cloudflare sans ouvrir une nouvelle page.
    """
    url = f'https://www.la-flamme-rouge.eu/maps/viewtrack/gpx/{race_id}'
    cookies = {c['name']: c['value'] for c in driver.get_cookies()}
    headers = {'User-Agent': driver.execute_script('return navigator.userAgent')}
    r = requests.get(url, cookies=cookies, headers=headers, allow_redirects=True)
    return r.content

In [ ]:
def create_driver():
    """Crée et retourne un driver undetected-chromedriver."""
    options = uc.ChromeOptions()
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--window-size=1920,1080')
    return uc.Chrome(options=options, headless=False, version_main=147)


def login(driver, username, password):
    """Passe Cloudflare et se connecte. Retourne True si succès."""
    driver.get('https://www.la-flamme-rouge.eu')
    print('⏳ Attente du challenge Cloudflare...')
    time.sleep(5)
    if 'Just a moment' in driver.title:
        print('⏳ Cloudflare encore actif, on attend encore...')
        time.sleep(10)
    print('✅ Cloudflare passé !')

    driver.get('https://www.la-flamme-rouge.eu/ucp.php?mode=login')
    WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.NAME, 'username')))

    username_field = driver.find_element(By.NAME, 'username')
    for char in username:
        username_field.send_keys(char)
        time.sleep(0.05)

    password_field = driver.find_element(By.NAME, 'password')
    for char in password:
        password_field.send_keys(char)
        time.sleep(0.05)

    login_btn = driver.find_element(By.NAME, 'login')
    driver.execute_script('arguments[0].click();', login_btn)
    time.sleep(3)

    if 'incorrect' in driver.page_source.lower():
        print('❌ Connexion échouée : identifiants incorrects.')
        return False

    print('✅ Connexion réussie !')
    return True


def scrape_calendar(driver, years=range(2017, 2026)):
    """Scrape le calendrier des courses pour les années données."""
    races = []
    for year in years:
        for month in range(1, 13):
            url = f'https://www.la-flamme-rouge.eu/maps/races/calendar?month={month}&year={year}'
            driver.get(url)
            time.sleep(2)
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            days = soup.find_all('div', {'class': 'day__body'})
            for day in days:
                for race in day.find_all('a', href=True):
                    try:
                        day_div = race.find_parent('td').find('div', {'class': 'day__header'})
                        race_day = int(day_div.find('div', {'class': 'day__header__day'}).text.strip())
                        race_name = str(year) + ' ' + ' '.join(race.find('div', {'class': 'race__name'}).text.strip().split())
                        race_id = race['href'].split('/')[-1].split('?')[0]
                        races.append([race_name, race_id, race_day, month, year])
                    except Exception as e:
                        print(f'Erreur parsing course : {e}')
            print(f'  📅 {month:02d}/{year} — {len(days)} jours trouvés')
    print(f'\n🏁 Total : {len(races)} courses récupérées.')
    return races

In [ ]:
import subprocess
subprocess.run(['pkill', '-f', 'undetected'], capture_output=True)
subprocess.run(['pkill', '-f', 'chromedriver'], capture_output=True)
print("✅ Processus nettoyés")

In [ ]:
# --- Étape 1 : Connexion ---
driver = create_driver()
if not login(driver, '...', '...''):
    driver.quit()

In [ ]:
# --- Étape 2 : Scraping du calendrier 2017-2025 ---
races = scrape_calendar(driver)

In [ ]:
races_complement = scrape_calendar(driver, years=[2025])
# Filtrer uniquement les mois manquants (sept-déc 2025)
races_complement = [r for r in races_complement if r[3] >= 9]
races = races + races_complement
print(f'Total après complément : {len(races)}')

In [ ]:
# --- Étape 3 : Téléchargement des GPX ---
os.makedirs(GPX_DIR, exist_ok=True)
for race in tqdm(races):
    race_name_clean = race[0].replace('"', '')
    output_path = os.path.join(GPX_DIR, race_name_clean.split('/')[0] + '.gpx')

    if not os.path.exists(output_path):
        gpx_content = get_gpx(race[1], driver)
        if b'Just a moment' not in gpx_content:
            with open(output_path, 'wb') as f:
                f.write(gpx_content)
        else:
            print(f'⚠️  Cloudflare bloqué sur : {race[0]}')

In [ ]:
# --- Étape 4 : Sauvegarde du calendrier ---
import pandas as pd
pd.DataFrame(races, columns=['name', 'id', 'day', 'month', 'year']).to_csv(
    '/Users/arthurdeletang/Desktop/Stage M1/Code/data/races.csv', index=False
)
print(f'✅ {len(races)} courses sauvegardées')

In [ ]:
# --- Diagnostic : fichiers par année ---
from collections import Counter
years = [f[:4] for f in os.listdir(GPX_DIR) if f.endswith('.gpx')]
counter = Counter(years)
for year in sorted(counter):
    print(f"{year} : {counter[year]} fichiers")

In [ ]:
# --- Diagnostic : courses manquantes ---
noms_presents = set(os.listdir(GPX_DIR))
manquants = [race for race in races if race[0].replace('"', '').split('/')[0] + '.gpx' not in noms_presents]
print(f'Courses totales  : {len(races)}')
print(f'Fichiers présents: {len(noms_presents)}')
print(f'Manquants        : {len(manquants)}')
for r in manquants[:20]:
    print(r)

In [ ]:
# --- Diagnostic : présence de l'élévation ---
GPX_DIR_2 = '/Users/arthurdeletang/Desktop/Stage M1/Code/data/gpx_files_2'
sans_elevation = []
avec_elevation = []

for f in os.listdir(GPX_DIR_2):
    if f.endswith('.gpx'):
        with open(os.path.join(GPX_DIR_2, f), 'r', encoding='utf-8') as file:
            content = file.read()
        if '<ele>' in content:
            avec_elevation.append(f)
        else:
            sans_elevation.append(f)

print(f'Avec élévation    : {len(avec_elevation)}')
print(f'Sans élévation    : {len(sans_elevation)}')
if sans_elevation:
    print('\nFichiers sans élévation :')
    for f in sans_elevation[:20]:
        print(f)

In [ ]:
# --- Étape 5 : Fermeture du driver ---
driver.quit()
print(f'GPX bruts     : {len(os.listdir(GPX_DIR))} fichiers dans {GPX_DIR}')

In [ ]:
import os

GPX_DIR = '...'

for fname in os.listdir(GPX_DIR):
    if ' / ' in fname:
        new_name = fname.replace(' / ', ' - ')
        os.rename(
            os.path.join(GPX_DIR, fname),
            os.path.join(GPX_DIR, new_name)
        )
        print(f'Renomme : {fname} -> {new_name}')